# Experiment Class Demo

The `Experiment` class:
- Stores data required to specify experiments, as well as possible intended analysis.
- Is meant to be instantiable with parital information, with the remaining fields progressively filled with the `replace` method.
- The `replace` method has validation checking to ensure that the data is well-formed:
    - It has various kinds of validation, and will warn the user and set certain properties to `None` if any other properties they depend on are changed. E.g. the default behaviour is to set `relations` to `None` if `paths` is changed.
    - This can be overridden by the caller (e.g. presumably stages will do this).

This notebook demonstrates the behavior of the `Experiment` class:
1. Properties and data storage
2. `replace` validation logic
3. Composing experiments with `+`
4. The `is_executable` property

In [ ]:
from qiskit_noise_learning.experiment_builder.experiment import Experiment
from qiskit_noise_learning.sequences import InstructionSequence, Path

## 1. Properties and Data Storage

All fields are optional. An `Experiment` can start empty and be progressively populated.

In [ ]:
# Create an empty experiment
exp = Experiment()
print(exp)

In [ ]:
# Create some simple instruction sequences (empty fragments for demo purposes)
unbound_seq_1 = InstructionSequence([], [], [])  # unbound (depth=None)
unbound_seq_2 = InstructionSequence([], [], [])  # unbound
bound_seq = InstructionSequence([], [], [], depth=3)  # bound at depth 3

# Create some simple paths
unbound_path = Path([], [], [])  # unbound
bound_path = Path([], [], [], depth=0)  # bound at depth 0

print(f"unbound_seq_1.is_unbound = {unbound_seq_1.is_unbound}")
print(f"bound_seq.is_unbound = {bound_seq.is_unbound}")
print(f"bound_seq.depth = {bound_seq.depth}")

In [ ]:
# Create an experiment with data populated
exp = Experiment(
    paths=[unbound_path, bound_path],
    instruction_sequences=[unbound_seq_1, unbound_seq_2, bound_seq],
    randomization_multipliers=[1, 2, 1],
    relations={(0, 0), (0, 1), (1, 2)},
    shots=1024,
    randomizations=50,
)
print(exp)

In [ ]:
# Access individual properties
print(f"shots: {exp.shots}")
print(f"randomizations: {exp.randomizations}")
print(f"randomization_multipliers: {exp.randomization_multipliers}")
print(f"relations: {exp.relations}")
print(f"fidelity_model: {exp.fidelity_model}")

## 2. Replace Validation Logic

`Experiment.replace` returns a new experiment with specified fields overridden. When
`validate=True` (default), it enforces several constraints.

### Co-replacement: instruction_sequences and randomization_multipliers

These two fields must always both be `None` or both be non-`None`.

In [ ]:
# This works: replacing both together
new_exp = exp.replace(
    instruction_sequences=[bound_seq, bound_seq],
    randomization_multipliers=[1, 1],
)
print(new_exp)

In [ ]:
# This fails: replacing instruction_sequences without randomization_multipliers
try:
    exp.replace(instruction_sequences=[bound_seq])
except ValueError as e:
    print(f"ValueError: {e}")

### Length consistency

`randomization_multipliers` must have the same length as `instruction_sequences`.

In [ ]:
try:
    exp.replace(
        instruction_sequences=[bound_seq, bound_seq],
        randomization_multipliers=[1, 1, 1],  # wrong length
    )
except ValueError as e:
    print(f"ValueError: {e}")

### Relations bounds checking

Setting `relations` requires `paths` and `instruction_sequences` to be present, with all indices in bounds.

In [ ]:
# Out-of-bounds relation
try:
    exp.replace(relations={(99, 0)})
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
# Setting relations on an experiment without paths
empty_exp = Experiment(
    instruction_sequences=[bound_seq],
    randomization_multipliers=[1],
)
try:
    empty_exp.replace(relations={(0, 0)})
except ValueError as e:
    print(f"ValueError: {e}")

### Soft invalidation

Replacing `paths` or `instruction_sequences` without providing new `relations` will
set `relations` to `None` with a warning.

In [ ]:
import warnings

print(f"Before: relations = {exp.relations}")

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    new_exp = exp.replace(
        paths=[bound_path],  # changing paths invalidates relations
    )
    print(f"After: relations = {new_exp.relations}")
    print(f"Warning: {w[0].message}")

### Bypassing validation

Set `validate=False` to skip all checks (useful in internal stage logic).

In [ ]:
# This would normally fail, but validate=False bypasses all checks
new_exp = exp.replace(validate=False, instruction_sequences=[bound_seq])
print(f"instruction_sequences length: {len(new_exp.instruction_sequences)}")
print(f"randomization_multipliers: {new_exp.randomization_multipliers}")

## 3. Composing Experiments with `+`

Two experiments can be added together. Scalar fields must match, list fields are
concatenated, and relation indices are offset appropriately.

In [ ]:
exp_a = Experiment(
    paths=[unbound_path],
    instruction_sequences=[bound_seq],
    randomization_multipliers=[1],
    relations={(0, 0)},
    shots=1024,
    randomizations=50,
)

exp_b = Experiment(
    paths=[bound_path],
    instruction_sequences=[bound_seq, bound_seq],
    randomization_multipliers=[2, 3],
    relations={(0, 0), (0, 1)},
    shots=1024,
    randomizations=50,
)

combined = exp_a + exp_b
print(combined)

In [ ]:
# Relations are offset: exp_b's relations are shifted by exp_a's list lengths
print(f"exp_a relations: {exp_a.relations}")
print(f"exp_b relations: {exp_b.relations}")
print(f"combined relations: {combined.relations}")
print(f"combined multipliers: {combined.randomization_multipliers}")

In [ ]:
# Scalar fields must match — mismatched shots causes an error
exp_c = Experiment(shots=512, randomizations=50)
try:
    exp_a + exp_c
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
# None + list: None is treated as empty (unless both are None)
exp_no_paths = Experiment(shots=1024, randomizations=50)
combined = exp_a + exp_no_paths
print(f"paths: {len(combined.paths)} (exp_a had 1 path, exp_no_paths had None)")

## 4. The `is_executable` Property

An experiment is executable when all instruction sequences are bound and complete,
and `randomization_multipliers` is set. `shots` and `randomizations` always have
default values (20 and 50), so they never block executability.

In [ ]:
# Not executable: has unbound sequences
exp_unbound = Experiment(
    instruction_sequences=[unbound_seq_1, bound_seq],
    randomization_multipliers=[1, 1],
    shots=1024,
    randomizations=50,
)
print(f"Has unbound sequences: is_executable = {exp_unbound.is_executable}")

In [ ]:
# Executable: all sequences bound and complete, all execution params set
exp_ready = Experiment(
    instruction_sequences=[bound_seq, bound_seq],
    randomization_multipliers=[1, 2],
    shots=1024,
    randomizations=50,
)
print(f"All set: is_executable = {exp_ready.is_executable}")

In [ ]:
# Not executable: no instruction sequences at all
exp_empty = Experiment(shots=1024, randomizations=50)
print(f"No sequences: is_executable = {exp_empty.is_executable}")